# Bonus 1 — SOLUTION: Advanced Data Cleaning & Preprocessing

> **Instructor note:** Complete solution. Share after participants attempt the exercises.

Steps with participant exercises: 1 (extract_pages), 2 (normalise), 4 (quality filter), 5 (near-duplicate detection).


In [ ]:
import os
import re
import warnings
warnings.filterwarnings('ignore')

import pdfplumber
import pandas as pd
import numpy as np
from pathlib import Path
from langdetect import detect, LangDetectException
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv('../.env')

DATA_DIR = Path('../data/sample_dkv')
pdf_files = sorted(DATA_DIR.glob('*.pdf'))
print(f'Found {len(pdf_files)} PDFs: {[p.name for p in pdf_files]}')


---
## Step 1 — Extract text with pdfplumber (structure-aware)

`pdfplumber` can extract tables as structured objects and gives per-page bounding boxes,  
making it easier to strip headers/footers by position.


In [ ]:
def extract_pages(pdf_path: Path) -> list[dict]:
    """Extract pages with text and any tables, filtering headers/footers by Y position."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        page_height = pdf.pages[0].height if pdf.pages else 800
        header_cutoff = page_height * 0.08   # top 8% = header
        footer_cutoff = page_height * 0.92   # bottom 8% = footer

        for page_num, page in enumerate(pdf.pages, 1):
            # Extract words filtered by Y position (exclude header/footer bands)
            words = [
                w['text'] for w in (page.extract_words() or [])
                if header_cutoff < float(w.get('top', 0)) < footer_cutoff
            ]
            body_text = ' '.join(words).strip()

            # Extract tables separately and convert to markdown
            tables_md = []
            for table in (page.extract_tables() or []):
                rows = [' | '.join(str(c or '') for c in row) for row in table if any(row)]
                if rows:
                    tables_md.append('\n'.join(rows))

            pages.append({
                'source': pdf_path.name,
                'page': page_num,
                'text': body_text,
                'tables': tables_md,
            })
    return pages

# Extract all PDFs
all_pages = []
for pdf in pdf_files:
    pages = extract_pages(pdf)
    all_pages.extend(pages)
    print(f'{pdf.name}: {len(pages)} pages, {sum(len(p["tables"]) for p in pages)} tables')

print(f'\nTotal: {len(all_pages)} pages extracted')


---
## Step 2 — Text normalisation

Insurance PDFs often contain OCR artifacts, ligatures, and inconsistent whitespace.


In [ ]:
# Common OCR / PDF extraction artifacts in insurance docs
_LIGATURES = {
    '\ufb01': 'fi', '\ufb02': 'fl', '\ufb03': 'ffi', '\ufb04': 'ffl',
    '\u2019': "'", '\u2018': "'", '\u201c': '"', '\u201d': '"',
    '\u2013': '-', '\u2014': '-', '\u00a0': ' ',
}

def normalise(text: str) -> str:
    # Replace ligatures and smart quotes
    for src, dst in _LIGATURES.items():
        text = text.replace(src, dst)
    # Collapse runs of whitespace (but keep newlines)
    text = re.sub(r'[ \t]+', ' ', text)
    # Remove lines that are just a number (page numbers left by positional filter)
    text = re.sub(r'(?m)^\s*\d{1,3}\s*$', '', text)
    # Collapse multiple blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# Apply normalisation
for p in all_pages:
    p['text'] = normalise(p['text'])

# Show before/after on a sample
sample = all_pages[0]
print(f'Page 1 of {sample["source"]} ({len(sample["text"])} chars after normalisation):')
print(sample['text'][:400])


---
## Step 3 — Language detection

DKV Belgium documents may be FR, NL, or EN.  
Knowing the language per chunk lets you route queries to the right embedding model  
or filter by language in a multilingual deployment.


In [ ]:
def detect_language(text: str) -> str:
    """Return ISO 639-1 code or 'unknown' for short/ambiguous text."""
    if len(text.split()) < 10:
        return 'unknown'
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'

for p in all_pages:
    p['language'] = detect_language(p['text'])

lang_counts = pd.Series([p['language'] for p in all_pages]).value_counts()
print('Language distribution across pages:')
print(lang_counts.to_string())


---
## Step 4 — Quality filtering

Short chunks (< 50 words) are usually section titles, captions, or extraction noise —  
they add no retrieval value and dilute embedding space.


In [ ]:
MIN_WORDS = 50

before = len(all_pages)
all_pages = [p for p in all_pages if len(p['text'].split()) >= MIN_WORDS]
print(f'Quality filter: {before} → {len(all_pages)} pages ({before - len(all_pages)} dropped)')

# Word count distribution
word_counts = [len(p['text'].split()) for p in all_pages]
print(f'Word count: min={min(word_counts)}, median={int(np.median(word_counts))}, max={max(word_counts)}')


---
## Step 5 — Near-duplicate detection

Boilerplate sections (legal disclaimers, contact info) appear on multiple pages.  
Duplicate chunks waste embedding space and skew retrieval toward repeated content.

Strategy: embed all chunks, then flag pairs with cosine similarity > 0.97.


In [ ]:
embed_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

texts = [p['text'][:512] for p in all_pages]  # truncate for speed
print(f'Embedding {len(texts)} pages...')
embeddings = embed_model.encode(texts, show_progress_bar=True, batch_size=32)

SIM_THRESHOLD = 0.97
sim_matrix = cosine_similarity(embeddings)
np.fill_diagonal(sim_matrix, 0)  # ignore self-similarity

duplicates = set()
for i in range(len(texts)):
    if i not in duplicates:
        near_dupes = np.where(sim_matrix[i] > SIM_THRESHOLD)[0]
        for j in near_dupes:
            if j > i:
                duplicates.add(j)

print(f'Near-duplicates found: {len(duplicates)}')
if duplicates:
    ex = list(duplicates)[0]
    orig = np.argmax(sim_matrix[ex])
    print(f'\nExample duplicate (page {ex}) vs original (page {orig}):')
    print(f'  [{orig}] {all_pages[orig]["source"]} p{all_pages[orig]["page"]}: {all_pages[orig]["text"][:120]}...')
    print(f'  [{ex}]  {all_pages[ex]["source"]} p{all_pages[ex]["page"]}: {all_pages[ex]["text"][:120]}...')

all_pages_deduped = [p for i, p in enumerate(all_pages) if i not in duplicates]
print(f'\nAfter deduplication: {len(all_pages)} → {len(all_pages_deduped)} pages')


---
## Step 6 — Compare retrieval quality: baseline vs cleaned

Index both the raw and cleaned chunks into separate ChromaDB collections,  
then compare retrieval precision on a few sample queries.


In [ ]:
import chromadb
import sys
sys.path.insert(0, '..')
from src.embedder import embed_texts

chroma = chromadb.PersistentClient(path='../chroma_db')

def index_pages(pages: list[dict], collection_name: str):
    try:
        chroma.delete_collection(collection_name)
    except Exception:
        pass
    col = chroma.create_collection(collection_name, metadata={'hnsw:space': 'cosine'})
    texts = [p['text'] for p in pages]
    embeddings = embed_model.encode(texts, batch_size=32, show_progress_bar=False).tolist()
    col.add(
        ids=[f'{p["source"]}::p{p["page"]}' for p in pages],
        documents=texts,
        embeddings=embeddings,
        metadatas=[{'source': p['source'], 'page': p['page'], 'language': p.get('language', 'unknown')} for p in pages],
    )
    return col

print('Indexing cleaned pages...')
col_clean = index_pages(all_pages_deduped, 'workshop_rag_clean')
print(f'Clean collection: {col_clean.count()} docs')

# Compare a query
test_query = 'What is covered under DKV hospitalisation insurance?'
q_emb = embed_model.encode([test_query]).tolist()

for name, col in [('workshop_rag', chroma.get_or_create_collection('workshop_rag', metadata={'hnsw:space': 'cosine'})),
                  ('workshop_rag_clean', col_clean)]:
    results = col.query(query_embeddings=q_emb, n_results=3, include=['documents', 'distances'])
    print(f'\n{name} top-3 results:')
    for doc, dist in zip(results['documents'][0], results['distances'][0]):
        print(f'  [{1-dist:.2f}] {doc[:100]}...')


---
## Summary

| Step | Effect |
|------|--------|
| Header/footer removal | Eliminates noise from positional extraction |
| Table extraction | Tables become searchable text rows |
| Normalisation | Fixes encoding/ligature artifacts |
| Quality filter | Removes short non-informative chunks |
| Deduplication | Prevents boilerplate dominating retrieval |

**Key takeaway:** Clean input → better embeddings → more precise retrieval.  
The evaluation metrics from Module 4 are the quantitative proof.
